In [ ]:
import json
from pathlib import Path

import altair as alt
import marimo as mo
import polars as pl

# Benchmark run

A corpus of synthetic documents with known planted values, and what a
pipeline reported finding in them.

Nothing here is scored. A detection is only *matched* to a planted value
by the scorer, which knows about span overlap, boundary tolerance, and
label equivalence. This shows the two side by side so the shape of a run
is visible before any of that is decided.

In [ ]:
# Repository root, from this file's location.
root = Path(__file__).resolve().parent.parent

runs = sorted(
    (path.parent for path in root.glob("runs/*/run.json")),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

run_picker = mo.ui.dropdown(
    options={path.name: path for path in runs},
    value=runs[0].name if runs else None,
    label="Run",
)
run_picker if runs else mo.md(
    "**No runs found.** Generate a corpus and run `bench` first."
)

&lt;marimo-dropdown data-initial-value=&#x27;[&amp;quot;3-mtrike2a&amp;quot;]&#x27; data-label=&#x27;&amp;quot;&amp;#92;u003cspan class=&amp;#92;&amp;quot;markdown prose dark:prose-invert contents&amp;#92;&amp;quot;&amp;#92;u003e&amp;#92;u003cspan class=&amp;#92;&amp;quot;paragraph&amp;#92;&amp;quot;&amp;#92;u003eRun&amp;#92;u003c/span&amp;#92;u003e&amp;#92;u003c/span&amp;#92;u003e&amp;quot;&#x27; data-options=&#x27;[&amp;quot;3-mtrike2a&amp;quot;]&#x27; data-allow-select-none=&#x27;false&#x27; data-searchable=&#x27;false&#x27; data-full-width=&#x27;false&#x27; data-disabled=&#x27;false&#x27;&gt;&lt;/marimo-dropdown&gt;

In [ ]:
# The dropdown has no value until it is touched, and it is never touched
# when the notebook is run as a script, so fall back to the newest run.
run_dir = run_picker.value or (runs[0] if runs else None)
if run_dir is None:
    raise FileNotFoundError(
        "No runs found. Generate a corpus and run `bench` first."
    )

# `json` rather than polars for these two: an index is one object read for
# scalar fields, not a table. `pl.read_json` would return a one-row frame to
# immediately index back out of. Everything row-shaped below is polars.
run = json.loads((run_dir / "run.json").read_text())

# The corpus the run was made against. A run records the seed and the
# manifest's digest, so a mismatch here means the corpus has been
# regenerated since — and the two are no longer comparable.
corpus_dir = root / "corpus"
manifest = json.loads((corpus_dir / "manifest.json").read_text())

In [ ]:
same_corpus = run["corpusSeed"] == manifest["seed"]
mismatch = (
    "" if same_corpus else f" ⚠️ the corpus on disk is seed {manifest['seed']}"
)

mo.md(
    f"""
    | | |
    |---|---|
    | Run | `{run["id"]}` |
    | Corpus seed | {run["corpusSeed"]}{mismatch} |
    | Pipeline | `{run["pipeline"]}` |
    | Records | {len(run["records"])} |
    | Labels in scope | {len(manifest["labels"])} |
    | Started | {run["startedAt"]} |
    """
)

| | |
|---|---|
| Run | `3-mtrike2a` |
| Corpus seed | 3 |
| Pipeline | `benchmark` |
| Records | 20 |
| Labels in scope | 21 |
| Started | 2026-09-07T17:27:49.042Z |

In [ ]:
def truth_frame() -> pl.DataFrame:
    """Every planted value, one row per occurrence.

    Read with polars so the whole notebook speaks one vocabulary: the shape
    of the data is declared here rather than assembled by hand, and it
    arrives in the frame the aggregations below already work in.

    Not for speed — a `json.loads` loop is measurably quicker at this size,
    since polars parses each file separately and pays for the concat on top.
    The gain is that the schema is visible and nothing walks dictionaries.

    The concat is diagonal because truth files differ in shape: an entity
    carries `expect` and `adversarial` only when it has them, so a uniform
    vstack rejects them.
    """
    paths = sorted(corpus_dir.glob("records/*/truth.json"))
    truth = pl.concat(
        [pl.read_json(path) for path in paths], how="diagonal_relaxed"
    )

    entities = (
        truth.select("id", "format", "specId", "entities")
        # Renamed before unnesting, since an entity has its own `id`.
        .rename({"id": "record"})
        .explode("entities")
        .unnest("entities")
        .rename({"id": "entityId"})
        .select(
            "record",
            "format",
            "specId",
            "entityId",
            "label",
            "expect",
            "adversarial",
        )
    )

    occurrences = (
        truth.select("id", "occurrences")
        .rename({"id": "record"})
        .explode("occurrences")
        .unnest("occurrences")
        .rename({"id": "occurrenceId"})
        .with_columns(
            # A tabular value is addressed by cell rather than by an offset
            # into the document, so it has no range and these are null.
            pl.col("location")
            .struct.field("ranges")
            .list.first()
            .struct.field("start")
            .alias("start"),
            pl.col("location")
            .struct.field("ranges")
            .list.first()
            .struct.field("end")
            .alias("end"),
        )
        .drop("location", "modalityId")
    )

    return occurrences.join(entities, on=["record", "entityId"], how="left")

def found_frame() -> pl.DataFrame:
    """Every detection, one row each, plus a row for a failed record."""
    paths = sorted((run_dir / "records").glob("*.json"))
    outcomes = pl.concat(
        [pl.read_json(path) for path in paths], how="diagonal_relaxed"
    )

    columns = [
        "record",
        "label",
        "confidence",
        "recognizer",
        "start",
        "end",
        "failed",
    ]

    detected = (
        outcomes.filter(pl.col("status") == "detected")
        .select("recordId", "detected")
        .explode("detected")
        .unnest("detected")
        .with_columns(
            pl.col("location")
            .struct.field("ranges")
            .list.first()
            .struct.field("start")
            .alias("start"),
            pl.col("location")
            .struct.field("ranges")
            .list.first()
            .struct.field("end")
            .alias("end"),
            pl.lit(None, dtype=pl.String).alias("failed"),
        )
        .rename({"recordId": "record"})
        .select(columns)
    )

    # A run with no failures has no `stage` column at all, since the concat
    # only sees the shape of the files that exist.
    if "stage" not in outcomes.columns:
        return detected

    failed = (
        outcomes.filter(pl.col("status") == "failed")
        .select("recordId", "stage")
        .rename({"recordId": "record", "stage": "failed"})
        .with_columns(
            pl.lit(None, dtype=pl.String).alias("label"),
            pl.lit(None, dtype=pl.Float64).alias("confidence"),
            pl.lit(None, dtype=pl.String).alias("recognizer"),
            pl.lit(None, dtype=pl.Int64).alias("start"),
            pl.lit(None, dtype=pl.Int64).alias("end"),
        )
        .select(columns)
    )

    return pl.concat([detected, failed], how="vertical")

planted = truth_frame()
# The format comes from the corpus, so a failed record still carries one.
found = found_frame().join(
    planted.select("record", "format").unique(), on="record", how="left"
)

/var/folders/2q/n2z04_697gsdkvzq5zdfy9fc0000gn/T/marimo_15871/__marimo__cell_PKri_.py:25: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("entities")
/var/folders/2q/n2z04_697gsdkvzq5zdfy9fc0000gn/T/marimo_15871/__marimo__cell_PKri_.py:42: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("occurrences")
/var/folders/2q/n2z04_697gsdkvzq5zdfy9fc0000gn/T/marimo_15871/__marimo__cell_PKri_.py:84: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("detected")


## By label

What was planted against what came back, per label. These are counts,
not a score: a detection is not attributed to a planted value here, so a
label with equal counts on both sides has not necessarily been matched
correctly.

In [ ]:
planted_by_label = planted.group_by("label").agg(
    pl.len().alias("planted"),
    (pl.col("expect") == "ignored").sum().alias("must_not_detect"),
)
found_by_label = (
    found.filter(pl.col("label").is_not_null())
    .group_by("label")
    .agg(pl.len().alias("detected"))
)

by_label = (
    planted_by_label.join(found_by_label, on="label", how="full", coalesce=True)
    .fill_null(0)
    .sort("planted", descending=True)
)
by_label

label,planted,must_not_detect,detected
str,u32,u32,u32
"""person_name""",50,0,0
"""bank_account""",32,18,4
"""email_address""",31,0,29
"""phone_number""",21,2,3
"""payment_card""",20,2,8
…,…,…,…
"""date_of_birth""",2,0,2
"""username""",2,0,0
"""mac_address""",2,0,1


In [ ]:
chart_data = by_label.unpivot(
    index="label",
    on=["planted", "detected"],
    variable_name="side",
    value_name="count",
).filter(pl.col("count") > 0)

mo.ui.altair_chart(
    alt.Chart(chart_data)
    .mark_bar()
    .encode(
        y=alt.Y("label:N", sort="-x", title=None),
        x=alt.X("count:Q", title="occurrences"),
        yOffset="side:N",
        color=alt.Color("side:N", title=None),
        tooltip=["label", "side", "count"],
    )
    .properties(height=alt.Step(12))
)

## By format

Where a pipeline reads a document differently, this is where it shows.
A format whose planted values never come back is usually one whose text
the pipeline did not extract, rather than one whose values it failed to
recognise.

In [ ]:
by_format = (
    planted.group_by("format")
    .agg(pl.len().alias("planted"), pl.col("record").n_unique().alias("records"))
    .join(
        found.filter(pl.col("label").is_not_null())
        .group_by("format")
        .agg(pl.len().alias("detected")),
        on="format",
        how="left",
    )
    .fill_null(0)
    .with_columns(
        (pl.col("detected") / pl.col("planted")).round(2).alias("ratio"),
    )
    .sort("planted", descending=True)
)
by_format

format,planted,records,detected,ratio
str,u32,u32,u32,f64
"""txt""",125,11,54,0.43
"""csv""",58,4,0,0.0
"""json""",39,3,17,0.44
"""xml""",26,2,14,0.54


## One record

The planted values and the detections for a single document, so a
disagreement can be looked at directly rather than inferred from counts.

In [ ]:
record_ids = planted.select(pl.col("record").unique().sort()).to_series().to_list()
record_picker = mo.ui.dropdown(
    options=record_ids,
    value=record_ids[0] if record_ids else None,
    label="Record",
)
record_picker

&lt;marimo-dropdown data-initial-value=&#x27;[&amp;quot;rec_0001&amp;quot;]&#x27; data-label=&#x27;&amp;quot;&amp;#92;u003cspan class=&amp;#92;&amp;quot;markdown prose dark:prose-invert contents&amp;#92;&amp;quot;&amp;#92;u003e&amp;#92;u003cspan class=&amp;#92;&amp;quot;paragraph&amp;#92;&amp;quot;&amp;#92;u003eRecord&amp;#92;u003c/span&amp;#92;u003e&amp;#92;u003c/span&amp;#92;u003e&amp;quot;&#x27; data-options=&#x27;[&amp;quot;rec_0001&amp;quot;,&amp;quot;rec_0002&amp;quot;,&amp;quot;rec_0003&amp;quot;,&amp;quot;rec_0004&amp;quot;,&amp;quot;rec_0005&amp;quot;,&amp;quot;rec_0006&amp;quot;,&amp;quot;rec_0007&amp;quot;,&amp;quot;rec_0008&amp;quot;,&amp;quot;rec_0009&amp;quot;,&amp;quot;rec_0010&amp;quot;,&amp;quot;rec_0011&amp;quot;,&amp;quot;rec_0012&amp;quot;,&amp;quot;rec_0013&amp;quot;,&amp;quot;rec_0014&amp;quot;,&amp;quot;rec_0015&amp;quot;,&amp;quot;rec_0016&amp;quot;,&amp;quot;rec_0017&amp;quot;,&amp;quot;rec_0018&amp;quot;,&amp;quot;rec_0019&amp;quot;,&amp;quot;rec_0020&amp;quot;]&#x27; data-allow-select-none=&#x27;false&#x27; data-searchable=&#x27;false&#x27; data-full-width=&#x27;false&#x27; data-disabled=&#x27;false&#x27;&gt;&lt;/marimo-dropdown&gt;

In [ ]:
mo.md("**Planted**")
planted.filter(pl.col("record") == record_picker.value).select(
    "label", "surface", "expect", "adversarial", "start", "end", "text"
).sort("start")

label,surface,expect,adversarial,start,end,text
str,str,str,str,i64,i64,str
"""organization_name""","""canonical""",null,null,25,36,"""Hagenes Inc"""
"""person_name""","""canonical""",null,"""common_word""",78,88,"""April Case"""
"""case_number""","""canonical""",null,"""format_collision""",102,113,"""471-88-2130"""
"""person_name""","""abbreviated""",null,"""common_word""",160,167,"""A. Case"""
"""ip_address""","""canonical""",null,"""unusual_context""",276,315,"""e80a:e3b3:610e:bb24:52dd:15f9:…"
…,…,…,…,…,…,…
"""email_address""","""canonical""",null,null,1396,1428,"""Ezequiel_Lehner31+jhaG@yahoo.c…"
"""person_name""","""misspelled""",null,"""common_word""",1509,1519,"""Apirl Case"""
"""case_number""","""canonical""","""ignored""","""format_collision""",1614,1628,"""CVE-2026-31337"""


In [ ]:
mo.md("**Detected**")
found.filter(pl.col("record") == record_picker.value).select(
    "label", "confidence", "recognizer", "start", "end", "failed"
).sort("start")

label,confidence,recognizer,start,end,failed
str,f64,str,i64,i64,str
"""government_id""",0.5,"""pattern""",102,113,null
"""ip_address""",0.6,"""pattern""",276,315,null
"""url""",0.5,"""pattern""",478,527,null
"""api_key""",0.95,"""pattern""",554,594,null
"""ip_address""",0.6,"""pattern""",687,703,null
"""monetary_amount""",0.55,"""pattern""",1180,1189,null
"""email_address""",0.5,"""pattern""",1396,1428,null


## Values that must survive

A corpus plants decoys — a catalog number shaped like an account, a
published switchboard, a public CVE — that a pipeline is expected to
leave alone. A detection overlapping one of these is over-redaction, and
the reason a benchmark that measures only recall is not a benchmark.

In [ ]:
planted.filter(pl.col("expect") == "ignored").select(
    "record", "label", "adversarial", "text"
)

record,label,adversarial,text
str,str,str,str
"""rec_0001""","""case_number""","""format_collision""","""CVE-2026-31337"""
"""rec_0001""","""device_id""","""format_collision""","""8.4.1-rc2+build.20260311"""
"""rec_0002""","""payment_card""","""format_collision""","""4111 1111 1111 1112"""
"""rec_0002""","""iban""","""format_collision""","""GB29 NWBK 6016 1331 9268 000"""
"""rec_0003""","""bank_account""","""unusual_context""","""000987654321"""
…,…,…,…
"""rec_0018""","""bank_account""","""format_collision""","""000555000555"""
"""rec_0018""","""bank_account""","""unusual_context""","""000987654321"""
"""rec_0018""","""bank_account""","""format_collision""","""000555000555"""
